# Mirror-CFE Validierung auf MNIST — Klassenpaar 7 vs. 9
**Datensatz:** MNIST (28×28, Graustufen)
**Modell:** ResNet-18 (1 Eingabekanal, von Grund auf trainiert — identisch zum FCVE-MNIST-Notebook)
**Methode:** Mirror Counterfactual Explanations (Mirror-CFE) — Spiegelung des Feature-Vektors
an der Entscheidungsgrenze (Position-Funktion, Eq. 1) + L-BFGS-Verfeinerung, KFE-Übergang
k=0→1, dekodiert über einen Bild-Decoder.

Dieses Notebook übernimmt **denselben ResNet-Klassifikator für 7 vs. 9** wie das
FCVE-MNIST-Notebook (`mnist-fcve-test.ipynb`) und wendet darauf die **Mirror-CFE-Methode**
an (analog zu `mirrorcfe-xray.ipynb` / `mirrorcfe-fire.ipynb`).

**Anpassung an MNIST:** Bei 28×28-Eingabe ist die letzte Conv-Feature-Map bereits räumlich
1×1 (vgl. Cell 6 Output) — es gibt also keine Zwischenschicht-Skips. Daher wird wie im
FCVE-MNIST-Notebook ein `SimpleDecoder` (reiner GAP-Vektor → Bild) statt des
Skip-Connection-Decoders (SSC/SPE) aus den 224×224-Röntgen-/Fire-Notebooks verwendet.
Die Mirror-Geometrie selbst (`position_function`, `compute_flk`, `refine_with_lbfgs`)
ist 1:1 aus `mirrorcfe-xray.ipynb` übernommen und funktioniert für 1×1-Maps unverändert.

## 1. Imports & Konfiguration

In [ ]:
import os
import struct
import numpy as np
from pathlib import Path
from array import array
from os.path import join
from tqdm import tqdm
from collections import OrderedDict
from functools import partial

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
from torch.utils.data import DataLoader, Dataset, random_split

torch.manual_seed(2024)
np.random.seed(2024)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Gerät:', DEVICE)

# ── Pfade (MNIST-Rohdaten, IDX-Format, wie auf Kaggle bereitgestellt) ─────────
INPUT_PATH = '/kaggle/input/datasets/hojjatk/mnist-dataset'
TRAIN_IMAGES_PATH = join(INPUT_PATH, 'train-images-idx3-ubyte/train-images-idx3-ubyte')
TRAIN_LABELS_PATH = join(INPUT_PATH, 'train-labels-idx1-ubyte/train-labels-idx1-ubyte')
TEST_IMAGES_PATH  = join(INPUT_PATH, 't10k-images-idx3-ubyte/t10k-images-idx3-ubyte')
TEST_LABELS_PATH  = join(INPUT_PATH, 't10k-labels-idx1-ubyte/t10k-labels-idx1-ubyte')
OUT_DIR = '/kaggle/working'

# ── Klassenpaar (Quelle 7, Ziel 9 — wie FCVE-Paper Fig. 3/4) ──────────────────
SOURCE_DIGIT = 7
TARGET_DIGIT = 9
# Klasse 0 = SOURCE_DIGIT, Klasse 1 = TARGET_DIGIT
CLASS_NAMES = {0: str(SOURCE_DIGIT), 1: str(TARGET_DIGIT)}

IMG_SIZE   = 28
BATCH_SIZE = 64

# Klassifikator-Hyperparameter (identisch zum FCVE-MNIST-Notebook)
CLF_LR     = 1e-3
CLF_EPOCHS = 10

# Mirror-CFE Hyperparameter
LBFGS_ITERS = 20      # L-BFGS-Verfeinerungsschritte (mirrorcfe-xray: num_iterations)

# Decoder-Hyperparameter
DEC_LR        = 1e-3
DEC_EPOCHS    = 15    # Phase 1: reine Rekonstruktion
DEC_FT_EPOCHS = 5     # Phase 2: Feinabstimmung auf KFE-/Mirror-Vektoren
DEC_FT_LR     = 2e-4
W_CLS         = 1.0   # Gewicht Klassifikations-Konsistenz (KLD) entlang des KFE-Pfads
W_FEA         = 0.1   # Gewicht Feature-Konsistenz entlang des KFE-Pfads

print(f'Klassenpaar: {SOURCE_DIGIT} (Klasse 0) vs. {TARGET_DIGIT} (Klasse 1)')


## 2. MNIST-Loader (IDX-Format)
Liest die rohen MNIST-Dateien direkt ein (kein torchvision-Download nötig — funktioniert
offline auf Kaggle, sofern der MNIST-Datensatz als Input hinzugefügt wurde).

In [ ]:
class MnistDataloader(object):
    """IDX-Format-Loader für MNIST (identisch zur vom Nutzer bereitgestellten Referenz)."""
    def __init__(self, training_images_filepath, training_labels_filepath,
                 test_images_filepath, test_labels_filepath):
        self.training_images_filepath = training_images_filepath
        self.training_labels_filepath = training_labels_filepath
        self.test_images_filepath = test_images_filepath
        self.test_labels_filepath = test_labels_filepath

    def read_images_labels(self, images_filepath, labels_filepath):
        labels = []
        with open(labels_filepath, 'rb') as file:
            magic, size = struct.unpack(">II", file.read(8))
            if magic != 2049:
                raise ValueError(f'Magic number mismatch, expected 2049, got {magic}')
            labels = array("B", file.read())

        with open(images_filepath, 'rb') as file:
            magic, size, rows, cols = struct.unpack(">IIII", file.read(16))
            if magic != 2051:
                raise ValueError(f'Magic number mismatch, expected 2051, got {magic}')
            image_data = array("B", file.read())
        images = []
        for i in range(size):
            images.append([0] * rows * cols)
        for i in range(size):
            img = np.array(image_data[i * rows * cols:(i + 1) * rows * cols])
            img = img.reshape(28, 28)
            images[i][:] = img

        return images, labels

    def load_data(self):
        x_train, y_train = self.read_images_labels(self.training_images_filepath,
                                                     self.training_labels_filepath)
        x_test, y_test = self.read_images_labels(self.test_images_filepath,
                                                  self.test_labels_filepath)
        return (x_train, y_train), (x_test, y_test)


mnist_dataloader = MnistDataloader(TRAIN_IMAGES_PATH, TRAIN_LABELS_PATH,
                                    TEST_IMAGES_PATH, TEST_LABELS_PATH)
(x_train_all, y_train_all), (x_test_all, y_test_all) = mnist_dataloader.load_data()

x_train_all = np.array(x_train_all, dtype=np.uint8)
y_train_all = np.array(y_train_all, dtype=np.int64)
x_test_all  = np.array(x_test_all,  dtype=np.uint8)
y_test_all  = np.array(y_test_all,  dtype=np.int64)

print(f'MNIST geladen — Train: {x_train_all.shape}  Test: {x_test_all.shape}')


## 3. Filtern auf das Klassenpaar & DataLoader
Nur Bilder der Quell- und Zielklasse werden behalten und auf {0,1} ummappt
(0 = Quellklasse 7, 1 = Zielklasse 9).

In [ ]:
def filter_and_remap(images, labels, source_digit, target_digit):
    """Behält nur Bilder von source_digit/target_digit, remapped Labels auf 0/1."""
    mask = (labels == source_digit) | (labels == target_digit)
    imgs_f = images[mask]
    lbls_f = labels[mask]
    remapped = np.where(lbls_f == source_digit, 0, 1).astype(np.int64)
    return imgs_f, remapped


x_train_f, y_train_f = filter_and_remap(x_train_all, y_train_all, SOURCE_DIGIT, TARGET_DIGIT)
x_test_f,  y_test_f  = filter_and_remap(x_test_all,  y_test_all,  SOURCE_DIGIT, TARGET_DIGIT)

print(f'Train gefiltert: {x_train_f.shape[0]} Bilder '
      f'(Klasse 0={int((y_train_f==0).sum())}, Klasse 1={int((y_train_f==1).sum())})')
print(f'Test  gefiltert: {x_test_f.shape[0]} Bilder '
      f'(Klasse 0={int((y_test_f==0).sum())}, Klasse 1={int((y_test_f==1).sum())})')


class MnistPairDataset(Dataset):
    """Liefert (Bild, Label, Dateiname) — Dateiname ist ein synthetischer Index-String,
    damit das Interface zu den Röntgen-/Fire-Notebooks identisch bleibt."""
    def __init__(self, images, labels, augment=False):
        self.images  = images
        self.labels  = labels
        self.augment = augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx].astype(np.float32) / 255.0   # [0,1]
        if self.augment and np.random.rand() < 0.5:
            img = np.fliplr(img).copy()
        img_t = torch.from_numpy(img).unsqueeze(0)           # (1, 28, 28)
        img_t = (img_t - 0.5) / 0.5                          # Normalisierung auf [-1, 1]
        label = int(self.labels[idx])
        fname = f'{idx:06d}.png'
        return img_t, label, fname


full_train_ds = MnistPairDataset(x_train_f, y_train_f, augment=True)
test_ds       = MnistPairDataset(x_test_f,  y_test_f,  augment=False)

# 90/10 Split des Trainingssatzes für Decoder-Validation (Test-Set bleibt unberührt)
n_val   = int(len(full_train_ds) * 0.1)
n_train = len(full_train_ds) - n_val
generator = torch.Generator().manual_seed(42)
train_split, val_split = random_split(full_train_ds, [n_train, n_val], generator=generator)

full_train_ds_eval = MnistPairDataset(x_train_f, y_train_f, augment=False)
val_dataset   = torch.utils.data.Subset(full_train_ds_eval, val_split.indices)
train_dataset = torch.utils.data.Subset(full_train_ds, train_split.indices)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,       batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

train_labels = [int(full_train_ds.labels[i]) for i in train_split.indices]
print(f'Train: {len(train_dataset)}  Val: {len(val_dataset)}  Test: {len(test_ds)}')

# ── Sichtprüfung ───────────────────────────────────────────────────────────────
imgs_chk, lbls_chk, _ = next(iter(train_loader))
fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
for i in range(8):
    img_np = (imgs_chk[i, 0].numpy() * 0.5) + 0.5
    axes[i].imshow(img_np, cmap='gray')
    axes[i].set_title(CLASS_NAMES[int(lbls_chk[i])], fontsize=10)
    axes[i].axis('off')
plt.suptitle('Stichprobe aus train_loader')
plt.tight_layout()
plt.show()


## 4. ResNet-18-Klassifikator definieren
Gleiche Architektur wie in den Röntgen-/Fire-Notebooks (1:1 übernommen), aber mit
`in_channels=1` (MNIST ist Graustufen) und von Grund auf trainiert. Die Struktur
`model.encoder` / `model.decoder.decoder` (linearer Kopf) ist identisch — damit
funktionieren die Mirror-Geometrie-Funktionen unverändert.

In [ ]:
class Conv2dAuto(nn.Conv2d):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.padding = (self.kernel_size[0] // 2, self.kernel_size[1] // 2)

conv3x3 = partial(Conv2dAuto, kernel_size=3, bias=False)

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.in_channels  = in_channels
        self.out_channels = out_channels
        self.blocks   = nn.Identity()
        self.shortcut = nn.Identity()
    def forward(self, x):
        residual = self.shortcut(x) if self.should_apply_shortcut else x
        x = self.blocks(x)
        x += residual
        return x
    @property
    def should_apply_shortcut(self):
        return self.in_channels != self.out_channels

class ResNetResidualBlock(ResidualBlock):
    def __init__(self, in_channels, out_channels, expansion=1,
                 downsampling=1, conv=conv3x3, *args, **kwargs):
        super().__init__(in_channels, out_channels)
        self.expansion    = expansion
        self.downsampling = downsampling
        self.conv         = conv
        self.shortcut = (
            nn.Sequential(OrderedDict({
                'conv': nn.Conv2d(self.in_channels, self.expanded_channels,
                                  kernel_size=1, stride=self.downsampling, bias=False),
                'bn':   nn.BatchNorm2d(self.expanded_channels)
            }))
            if self.should_apply_shortcut else None
        )
    @property
    def expanded_channels(self):
        return self.out_channels * self.expansion
    @property
    def should_apply_shortcut(self):
        return self.in_channels != self.expanded_channels

def conv_bn(in_channels, out_channels, conv, *args, **kwargs):
    return nn.Sequential(OrderedDict({
        'conv': conv(in_channels, out_channels, *args, **kwargs),
        'bn':   nn.BatchNorm2d(out_channels)
    }))

class ResNetBasicBlock(ResNetResidualBlock):
    expansion = 1
    def __init__(self, in_channels, out_channels, activation=nn.ReLU, *args, **kwargs):
        super().__init__(in_channels, out_channels, *args, **kwargs)
        self.blocks = nn.Sequential(
            conv_bn(self.in_channels, self.out_channels,
                    conv=self.conv, bias=False, stride=self.downsampling),
            activation(),
            conv_bn(self.out_channels, self.expanded_channels,
                    conv=self.conv, bias=False),
        )

class ResNetLayer(nn.Module):
    def __init__(self, in_channels, out_channels, block=ResNetBasicBlock,
                 n=1, *args, **kwargs):
        super().__init__()
        downsampling = 2 if in_channels != out_channels else 1
        self.blocks = nn.Sequential(
            block(in_channels, out_channels, *args, **kwargs,
                  downsampling=downsampling),
            *[block(out_channels * block.expansion, out_channels,
                    downsampling=1, *args, **kwargs) for _ in range(n - 1)]
        )
    def forward(self, x):
        return self.blocks(x)

class ResNetEncoder(nn.Module):
    def __init__(self, in_channels=3, blocks_sizes=(64, 128, 256, 512),
                 depths=(2, 2, 2, 2), activation=nn.ReLU,
                 block=ResNetBasicBlock, *args, **kwargs):
        super().__init__()
        self.blocks_sizes = blocks_sizes
        self.gate = nn.Sequential(
            nn.Conv2d(in_channels, blocks_sizes[0], kernel_size=7,
                      stride=2, padding=3, bias=False),
            nn.BatchNorm2d(blocks_sizes[0]),
            activation(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )
        self.in_out_block_sizes = list(zip(blocks_sizes, blocks_sizes[1:]))
        self.blocks = nn.ModuleList([
            ResNetLayer(blocks_sizes[0], blocks_sizes[0], n=depths[0],
                        activation=activation, block=block, *args, **kwargs),
            *[ResNetLayer(in_ch * block.expansion, out_ch, n=n,
                          activation=activation, block=block, *args, **kwargs)
              for (in_ch, out_ch), n in zip(self.in_out_block_sizes, depths[1:])]
        ])
    def forward(self, x):
        x = self.gate(x)
        for block in self.blocks:
            x = block(x)
        return x

class ResNetDecoder(nn.Module):
    def __init__(self, in_features, n_classes):
        super().__init__()
        self.avg     = nn.AdaptiveAvgPool2d((1, 1))
        self.decoder = nn.Linear(in_features, n_classes)
    def forward(self, x):
        x = self.avg(x)
        x = x.view(x.size(0), -1)
        x = self.decoder(x)
        return x

class ResNet(nn.Module):
    def __init__(self, in_channels, n_classes, *args, **kwargs):
        super().__init__()
        self.encoder = ResNetEncoder(in_channels, *args, **kwargs)
        self.decoder = ResNetDecoder(
            self.encoder.blocks[-1].blocks[-1].expanded_channels, n_classes
        )
    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

def build_classifier():
    # in_channels=1 (Graustufen) -- einziger struktureller Unterschied zu den RGB-Notebooks
    return ResNet(in_channels=1, n_classes=2,
                  block=ResNetBasicBlock, depths=[2, 2, 2, 2])

classifier = build_classifier().to(DEVICE)
n_params = sum(p.numel() for p in classifier.parameters())
print(f'ResNet-18 (1 Eingabekanal) Parameter: {n_params:,}')

with torch.no_grad():
    dummy = torch.zeros(2, 1, IMG_SIZE, IMG_SIZE).to(DEVICE)
    out = classifier.encoder(dummy)
    print('Letzte Conv-Feature-Map Shape:', out.shape)   # erwartet (2, 512, 1, 1) bei 28x28


## 5. Klassifikator-Training

In [ ]:
clf_optimizer = torch.optim.Adam(classifier.parameters(), lr=CLF_LR)
clf_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    clf_optimizer, mode='min', factor=0.5, patience=2)
clf_ce_loss = nn.CrossEntropyLoss()

clf_train_losses, clf_val_losses, clf_val_accs = [], [], []
best_val_acc = 0.0

print('Starte Klassifikator-Training...')
print(f'Epochs: {CLF_EPOCHS}  LR: {CLF_LR}  Batch-Size: {BATCH_SIZE}')
print('-' * 60)

for epoch in range(1, CLF_EPOCHS + 1):
    classifier.train()
    total_loss, n_correct, n_total = 0.0, 0, 0
    for images, labels, _ in tqdm(train_loader, desc=f'Ep {epoch:02d}/{CLF_EPOCHS} Train',
                                   leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        logits = classifier(images)
        loss   = clf_ce_loss(logits, labels)

        clf_optimizer.zero_grad()
        loss.backward()
        clf_optimizer.step()

        total_loss += loss.item() * images.size(0)
        n_correct  += (logits.argmax(1) == labels).sum().item()
        n_total    += images.size(0)

    train_loss = total_loss / n_total
    train_acc  = n_correct / n_total
    clf_train_losses.append(train_loss)

    classifier.eval()
    val_loss_sum, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels, _ in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits = classifier(images)
            val_loss_sum += clf_ce_loss(logits, labels).item() * images.size(0)
            val_correct  += (logits.argmax(1) == labels).sum().item()
            val_total    += images.size(0)

    val_loss = val_loss_sum / val_total
    val_acc  = val_correct / val_total
    clf_val_losses.append(val_loss)
    clf_val_accs.append(val_acc)
    clf_scheduler.step(val_loss)

    print(f'Ep {epoch:02d}/{CLF_EPOCHS}  '
          f'Train Loss={train_loss:.4f} Acc={train_acc:.4f}  '
          f'Val Loss={val_loss:.4f} Acc={val_acc:.4f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'model_state_dict': classifier.state_dict(),
            'val_acc': val_acc,
            'class_names': CLASS_NAMES,
            'source_digit': SOURCE_DIGIT,
            'target_digit': TARGET_DIGIT,
        }, os.path.join(OUT_DIR, 'mnist_classifier.pth'))

print(f'\nBester Val-Acc: {best_val_acc:.4f} — Klassifikator gespeichert ✓')

# ── Test-Set-Accuracy (echtes Holdout) ────────────────────────────────────────
classifier.eval()
test_correct, test_total = 0, 0
with torch.no_grad():
    for images, labels, _ in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        logits = classifier(images)
        test_correct += (logits.argmax(1) == labels).sum().item()
        test_total   += images.size(0)
print(f'Test-Set Accuracy ({SOURCE_DIGIT} vs {TARGET_DIGIT}): {test_correct/test_total:.4f}')

# Klassifikator für Mirror-CFE/Decoder einfrieren
for param in classifier.parameters():
    param.requires_grad = False
classifier.eval()
print('Klassifikator eingefroren ✓')


## 6. Feature-Extraktion (Encoder-Hook)
Liefert die letzte Conv-Feature-Map (bei MNIST 1×1 räumlich) sowie den GAP-Vektor.
`classify_from_gap` reicht einen (ggf. modifizierten) GAP-Vektor durch den linearen Kopf.

In [ ]:
def extract_features(model, images):
    """
    Gibt zurück:
      fmaps : (B, 512, 1, 1) -- bei 28x28 MNIST-Input ist die letzte Feature-Map 1x1 räumlich
      z_gap : (B, 512)       -- nach Global Average Pooling
    """
    fmaps_list = []
    hook = model.encoder.blocks[-1].register_forward_hook(
        lambda m, i, o: fmaps_list.append(o)
    )
    with torch.no_grad():
        _ = model(images)
    hook.remove()
    fmaps = fmaps_list[0]
    z_gap = F.adaptive_avg_pool2d(fmaps, (1, 1)).flatten(1)
    return fmaps, z_gap


def classify_from_gap(classifier, z_gap_modified):
    """Klassifiziert einen (ggf. modifizierten) GAP-Vektor über den linearen Kopf."""
    return classifier.decoder.decoder(z_gap_modified)


with torch.no_grad():
    dummy = torch.zeros(2, 1, IMG_SIZE, IMG_SIZE).to(DEVICE)
    fm, zg = extract_features(classifier, dummy)
    print('Feature Maps Shape:', fm.shape)
    print('GAP Vector Shape:  ', zg.shape)


## 7. Mirror-CFE Kernfunktionen (Geometrie)
1:1 aus `mirrorcfe-xray.ipynb` übernommen. Die Entscheidungsgrenze wird als
Wm = W[1]-W[0], bm = b[1]-b[0] (zeigt Richtung Zielklasse 1) bestimmt.
`position_function` (Paper Eq. 1) spiegelt den Feature-Vektor an dieser Grenze;
bei k=1 liegt die volle Reflexion vor. `refine_with_lbfgs` verfeinert den
reflektierten Vektor, bis der Kopf die Zielklasse ausgibt.

In [ ]:
def get_boundary_params(model):
    """Entscheidungsgrenze: Wm = W[1] - W[0], bm = b[1] - b[0] (Richtung Klasse 1)."""
    W  = model.decoder.decoder.weight.data   # (2, 512)
    b  = model.decoder.decoder.bias.data     # (2,)
    Wm = W[1] - W[0]                         # (512,)
    bm = b[1] - b[0]                         # scalar
    return Wm, bm

def position_function(zs_batch, Wm, bm, k):
    """Paper Eq. 1: P(zs, Wm, bm, k) = zs - 2k*(Wm^T*zs+bm)*Ŵm  (auf Feature-Map)."""
    B, C, H, W = zs_batch.shape
    W_hat   = Wm / (Wm.norm() + 1e-8)
    zs_flat = zs_batch.view(B, C, -1)
    dot     = (Wm.view(1, C, 1) * zs_flat).sum(dim=1, keepdim=True)
    scalar  = dot + bm.item()
    delta   = 2 * k * scalar * W_hat.view(1, C, 1)
    return (zs_flat - delta).view(B, C, H, W)

def compute_flk(f4_s, zs, Wm, bm, k):
    """KFE-Feature-Map f^l_k via gleichmäßigem Kanal-Shift, sodass GAP(f^l_k) = z_k gilt
    (Supp. Eq. 17). Bei MNIST ist f4_s 1x1, daher flk == zk (bis auf Form)."""
    B, C, H, Wd = f4_s.shape
    W_hat = Wm / (Wm.norm() + 1e-8)
    dot   = (Wm.view(1, C) * zs).sum(1, keepdim=True) + bm
    z_delta = -2.0 * k * dot * W_hat.view(1, C)
    zk  = zs + z_delta
    flk = f4_s + z_delta.view(B, C, 1, 1)
    return flk, zk


def refine_with_lbfgs(model, zr_init, source_labels, cfe_labels,
                      orig_probs_2cls, num_iterations=20):
    """L-BFGS-Verfeinerung, bis der Kopf die Zielklasse (vertauschte Probs) ausgibt."""
    swapped = orig_probs_2cls.clone().detach()
    idx     = torch.arange(len(cfe_labels))
    src     = source_labels.view(-1)
    cfe     = cfe_labels.view(-1)
    tmp               = swapped[idx, cfe].clone()
    swapped[idx, cfe] = swapped[idx, src]
    swapped[idx, src] = tmp

    z         = Variable(zr_init.clone().detach(), requires_grad=True)
    optimizer = torch.optim.LBFGS([z], lr=0.1)

    def closure():
        optimizer.zero_grad()
        pooled = F.adaptive_avg_pool2d(z, (1, 1))
        flat   = torch.flatten(pooled, 1)
        logits = model.decoder.decoder(flat)
        probs  = torch.softmax(logits, dim=1)
        loss   = torch.norm(probs - swapped) ** 2
        loss.backward()
        return loss

    for _ in range(num_iterations):
        optimizer.step(closure)
    return z

def compute_mirror_cfe(model, images, device, num_iterations=20):
    """Vollständige Mirror-CFE Pipeline: Spiegelung (k=1) + L-BFGS-Verfeinerung.
    Rückgabe: mirror_fv (B,512,1,1), cfe_labels, source_labels, orig_probs."""
    model.eval()
    images = images.to(device)
    with torch.no_grad():
        logits        = model(images)
        probs         = torch.softmax(logits, dim=1)
        orig_probs    = probs[:, 1]              # P(Klasse 1)
        source_labels = logits.argmax(dim=1)
        cfe_labels    = 1 - source_labels

    zs_map, _ = extract_features(model, images)  # (B,512,1,1)
    Wm, bm = get_boundary_params(model)
    Wm, bm = Wm.to(device), bm.to(device)

    with torch.no_grad():
        zr_geometric = position_function(zs_map.clone(), Wm, bm, k=1.0)

    mirror_fv = refine_with_lbfgs(
        model, zr_geometric, source_labels, cfe_labels,
        probs.clone(), num_iterations=num_iterations
    )
    return mirror_fv, cfe_labels, source_labels, orig_probs

# Grenze einmal global bestimmen (für Decoder-Feinabstimmung & Visualisierung)
Wm, bm = get_boundary_params(classifier)
Wm, bm = Wm.to(DEVICE), bm.to(DEVICE)

def predict_from_features(model, feature_maps):
    """Gibt P(Klasse 1) für eine Feature-Map (B,512,1,1) zurück."""
    with torch.no_grad():
        pooled = F.adaptive_avg_pool2d(feature_maps, (1, 1))
        flat   = torch.flatten(pooled, 1)
        probs  = torch.softmax(model.decoder.decoder(flat), dim=1)[:, 1]
    return probs

print('Mirror-CFE Funktionen definiert ✓')


## 8. SimpleDecoder (kein Skip — 1×1 Features)
Da MNIST-Features 1×1 sind, gibt es keine Skip-Connections wie in den 224×224-Notebooks.
Der `SimpleDecoder` rekonstruiert das 28×28-Bild allein aus dem 512-dim GAP-Vektor
(identisch zum FCVE-MNIST-Decoder).

In [ ]:
class SimpleDecoder(nn.Module):
    """GAP-Vektor (512) -> 28x28 Graustufenbild. Kein Skip-Connection-Mechanismus nötig,
    da die MNIST-Feature-Map bereits 1x1 ist."""
    def __init__(self, latent_dim=512):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 256 * 7 * 7)
        self.up1 = self._up_block(256, 128)   # 7  -> 14
        self.up2 = self._up_block(128, 64)    # 14 -> 28
        self.out_conv = nn.Sequential(
            nn.Conv2d(64, 1, kernel_size=3, padding=1),
            nn.Tanh()
        )

    def _up_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )

    def forward(self, z_gap):
        x = self.fc(z_gap)
        x = x.view(-1, 256, 7, 7)
        x = self.up1(x)
        x = self.up2(x)
        return self.out_conv(x)


decoder = SimpleDecoder(latent_dim=512).to(DEVICE)
with torch.no_grad():
    dummy_z = torch.zeros(2, 512).to(DEVICE)
    out = decoder(dummy_z)
    print('Decoder Output:', out.shape)   # erwartet (2, 1, 28, 28)
print(f'Decoder Parameter: {sum(p.numel() for p in decoder.parameters()):,}')


## 9. Phase 1 — Decoder-Training (reine Rekonstruktion)
Der Decoder lernt zunächst, echte GAP-Vektoren in ihr Originalbild zurückzuübersetzen.

In [ ]:
dec_optimizer = torch.optim.Adam(decoder.parameters(), lr=DEC_LR)
dec_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(dec_optimizer, T_max=DEC_EPOCHS)
mae_loss = nn.L1Loss()

def denormalise(tensor):
    """[-1,1] -> [0,1]"""
    return (tensor * 0.5 + 0.5).clamp(0, 1)

def tanh_to_img(tensor):
    """Decoder-Output [-1,1] -> [0,1]"""
    return (tensor + 1.0) / 2.0

dec_train_losses, dec_val_losses = [], []

print('Starte Decoder-Training (reine Rekonstruktion)...')
print(f'Epochs: {DEC_EPOCHS}  LR: {DEC_LR}  Batch-Size: {BATCH_SIZE}')
print('-' * 60)

for epoch in range(1, DEC_EPOCHS + 1):
    decoder.train()
    total_loss, n_total = 0.0, 0

    for images, labels, _ in tqdm(train_loader, desc=f'Ep {epoch:02d}/{DEC_EPOCHS} Train',
                                leave=False):
        images = images.to(DEVICE)
        _, z_gap = extract_features(classifier, images)

        recon = decoder(z_gap)
        loss  = mae_loss(tanh_to_img(recon), denormalise(images))

        dec_optimizer.zero_grad()
        loss.backward()
        dec_optimizer.step()

        total_loss += loss.item() * images.size(0)
        n_total    += images.size(0)

    train_loss = total_loss / n_total
    dec_train_losses.append(train_loss)
    dec_scheduler.step()

    decoder.eval()
    val_loss_sum, val_total = 0.0, 0
    with torch.no_grad():
        for images, labels, _ in val_loader:
            images = images.to(DEVICE)
            _, z_gap = extract_features(classifier, images)
            recon = decoder(z_gap)
            val_loss_sum += mae_loss(tanh_to_img(recon), denormalise(images)).item() * images.size(0)
            val_total    += images.size(0)

    val_loss = val_loss_sum / val_total
    dec_val_losses.append(val_loss)
    print(f'Ep {epoch:02d}/{DEC_EPOCHS}  Train MAE={train_loss:.4f}  Val MAE={val_loss:.4f}')

torch.save({
    'model_state_dict': decoder.state_dict(),
    'train_losses': dec_train_losses,
    'val_losses': dec_val_losses,
    'dataset': 'mnist',
}, os.path.join(OUT_DIR, 'mirror_decoder_mnist.pth'))
print(f'\nDecoder gespeichert → {OUT_DIR}/mirror_decoder_mnist.pth ✓')


## 10. Phase 2 — Decoder-Feinabstimmung auf KFE-/Mirror-Vektoren
**Problem:** Der Decoder aus Phase 1 sah nur echte GAP-Vektoren `zs`. Entlang des
KFE-Pfads (Spiegelung Richtung Zielklasse) und insbesondere am reflektierten Vektor
(k=1) liegen die Eingaben außerhalb dieser Verteilung — ohne Feinabstimmung entstünden
diffuse Artefakte.

**Lösung (angelehnt an `mirror-decoder-xray.ipynb`, vereinfacht für 1×1-Features):**
Pro Batch wird ein zufälliges k∈[0,1] gezogen und der Decoder auf vier Zielen trainiert:
- **L_base** (k=0): `decoder(zs)` ≈ Originalbild (Rekonstruktion bleibt erhalten),
- **L_cf** (k=1): `decoder(z_cf)` ≈ echtes Bild der Zielklasse (geometrische Reflexion als CFE),
- **L_cls**: dekodiertes KFE-Bild soll laut Klassifikator die durch `z_k` implizierte Klasse
  zeigen (KLD, Mirror-Paper Eq. 2),
- **L_fea**: GAP-Features des dekodierten KFE-Bildes ≈ `z_k` (Feature-Konsistenz, Eq. 5).

Die geometrische Reflexion (`compute_flk`, k=1) genügt hier als CFE-Ziel — bei einem
linearen Kopf kippt die Reflexion an der Grenze die Klasse bereits; die teure
L-BFGS-Verfeinerung bleibt der Inferenz vorbehalten (wie in `mirror-decoder-xray.ipynb`).

In [ ]:
def renorm_mnist(img01):
    """[0,1] -> [-1,1] (Eingabe-Normalisierung des Klassifikators)."""
    return (img01 - 0.5) / 0.5

def gap_features_grad(model, images):
    """Wie extract_features, aber MIT Gradientenfluss (kein no_grad) — für L_fea/L_cls.
    Der Klassifikator ist eingefroren; Gradienten fließen nur durch die Aktivierungen
    zurück zum dekodierten Bild."""
    feats = []
    h = model.encoder.blocks[-1].register_forward_hook(lambda m, i, o: feats.append(o))
    _ = model(images)
    h.remove()
    return F.adaptive_avg_pool2d(feats[0], (1, 1)).flatten(1)

def loss_cls_path(xk01, zk):
    """KLD(σ(F(decoder)), σ(Kopf(z_k))) — dekodiertes Bild soll wie z_k klassifizieren."""
    logp  = F.log_softmax(classifier(renorm_mnist(xk01)), dim=1)
    p_tgt = torch.softmax(classify_from_gap(classifier, zk), dim=1).detach()
    return F.kl_div(logp, p_tgt, reduction='batchmean')

def loss_fea_path(xk01, zk):
    """||z_k - GAP(F(decoder))|| — Feature-Konsistenz."""
    zk_re = gap_features_grad(classifier, renorm_mnist(xk01))
    return (zk - zk_re).norm(dim=1).mean()

def pick_reference(images, labels, want_cls):
    """Wählt pro Sample ein echtes Referenzbild der Klasse want_cls aus dem Batch.
    Fallback: ein beliebiges anderes Sample."""
    B = images.size(0)
    idx_out = torch.empty(B, dtype=torch.long)
    for i in range(B):
        cand = (labels == want_cls[i]).nonzero(as_tuple=True)[0]
        cand = cand[cand != i]
        if len(cand) == 0:
            cand = torch.arange(B)[torch.arange(B) != i]
        idx_out[i] = cand[torch.randint(len(cand), (1,))]
    return images[idx_out]


dec_ft_optimizer = torch.optim.Adam(decoder.parameters(), lr=DEC_FT_LR)
dec_ft_losses = []

print('Starte Decoder-Feinabstimmung auf KFE-/Mirror-Vektoren...')
print(f'Epochs: {DEC_FT_EPOCHS}  LR: {DEC_FT_LR}  Batch-Size: {BATCH_SIZE}')
print('-' * 60)

for epoch in range(1, DEC_FT_EPOCHS + 1):
    decoder.train()
    total_loss, n_total = 0.0, 0

    for images, labels, _ in tqdm(train_loader, desc=f'Ep {epoch:02d}/{DEC_FT_EPOCHS} FT',
                                leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        B = images.size(0)

        with torch.no_grad():
            f4_s, zs = extract_features(classifier, images)
            src = classify_from_gap(classifier, zs).argmax(1)
            tgt = 1 - src
            k   = float(torch.rand(1).item())
            _, zk   = compute_flk(f4_s, zs, Wm, bm, k)    # KFE-Vektor bei zufälligem k
            _, z_cf = compute_flk(f4_s, zs, Wm, bm, 1.0)  # volle Reflexion (CFE)

        # L_base (k=0): Rekonstruktion des Originals bleibt erhalten
        loss_base = mae_loss(tanh_to_img(decoder(zs)), denormalise(images))

        # L_cf (k=1): CFE soll wie ein echtes Zielklassenbild aussehen
        target_imgs = pick_reference(images, labels, tgt).to(DEVICE)
        loss_cf = mae_loss(tanh_to_img(decoder(z_cf)), denormalise(target_imgs))

        # L_cls + L_fea: Treue entlang des KFE-Pfads (zufälliges k)
        xk01     = tanh_to_img(decoder(zk))
        loss_cls = loss_cls_path(xk01, zk)
        loss_fea = loss_fea_path(xk01, zk)

        loss = loss_base + loss_cf + W_CLS * loss_cls + W_FEA * loss_fea

        dec_ft_optimizer.zero_grad()
        loss.backward()
        dec_ft_optimizer.step()

        total_loss += loss.item() * B
        n_total    += B

    epoch_loss = total_loss / n_total
    dec_ft_losses.append(epoch_loss)
    print(f'Ep {epoch:02d}/{DEC_FT_EPOCHS}  Combined Loss={epoch_loss:.4f}  '
          f'(base + cf + {W_CLS}*cls + {W_FEA}*fea)')

torch.save({
    'model_state_dict': decoder.state_dict(),
    'finetune_losses': dec_ft_losses,
    'dataset': 'mnist',
}, os.path.join(OUT_DIR, 'mirror_decoder_mnist_finetuned.pth'))
print(f'\nFeinabgestimmter Decoder gespeichert → {OUT_DIR}/mirror_decoder_mnist_finetuned.pth ✓')


## 11. Trainingskurven

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(range(1, len(dec_train_losses)+1), dec_train_losses, 'o-', label='Train')
axes[0].plot(range(1, len(dec_val_losses)+1),   dec_val_losses,   'o-', label='Val')
axes[0].set_title('Phase 1 — Decoder Rekonstruktion (MAE)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MAE')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(range(1, len(dec_ft_losses)+1), dec_ft_losses, 'o-', color='darkorange')
axes[1].set_title('Phase 2 — KFE-Feinabstimmung (combined)')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 12. Decoder-Qualitätscheck — reine Rekonstruktion

In [ ]:
decoder.eval()
sample_imgs, sample_lbls, _ = next(iter(val_loader))
sample_imgs = sample_imgs[:8].to(DEVICE)

with torch.no_grad():
    _, z_gap = extract_features(classifier, sample_imgs)
    recon = decoder(z_gap)

fig, axes = plt.subplots(2, 8, figsize=(20, 5))
for i in range(8):
    orig_np  = denormalise(sample_imgs[i, 0].cpu()).numpy()
    recon_np = tanh_to_img(recon[i, 0].detach().cpu()).numpy().clip(0, 1)
    axes[0, i].imshow(orig_np, cmap='gray');  axes[0, i].axis('off')
    axes[1, i].imshow(recon_np, cmap='gray'); axes[1, i].axis('off')
    axes[0, i].set_title(CLASS_NAMES[int(sample_lbls[i])], fontsize=9)
axes[0, 0].set_ylabel('Original',       fontsize=11)
axes[1, 0].set_ylabel('Rekonstruktion', fontsize=11)
plt.suptitle('Decoder-Rekonstruktion (feinabgestimmt)', fontsize=12)
plt.tight_layout()
plt.show()


## 13. Mirror-CFE Visualisierung — KFE-Übergang (k=0 → k=1)
Analog zur Visualisierung in `mirrorcfe-xray.ipynb`, aber da auf MNIST ein Bild-Decoder
vorliegt, werden die **dekodierten Bilder** entlang des KFE-Pfads gezeigt
(statt Feature-Map-Heatmaps): k=0 = Quelle (7), k=0.5 = Entscheidungsgrenze,
k=1 = Reflexion (9). Die letzte Spalte zeigt die Konfidenz-Journey P(Klasse 1).

In [ ]:
def generate_kfe_transitions(zs, zr, k_steps=None):
    """KFE-Übergänge: lineare Interpolation von zs nach zr (GAP-Vektoren)."""
    if k_steps is None:
        k_steps = [0.0, 0.25, 0.5, 0.6, 0.75, 1.0]
    transitions = []
    with torch.no_grad():
        for k in k_steps:
            transitions.append((1 - k) * zs + k * zr.detach())
    return transitions, k_steps


def visualise_mirror_cfe(model, decoder, images, labels,
                         mirror_fv, cfe_labels, source_labels, orig_probs,
                         class_names, n_samples=4, save_path='mirror_cfe_mnist.png'):
    model.eval(); decoder.eval()
    images = images.to(DEVICE)
    n      = min(n_samples, images.size(0))

    _, zs_vec = extract_features(model, images[:n])          # (n,512)
    zr_vec    = mirror_fv[:n].flatten(1).detach()            # (n,512)

    k_steps = [0.0, 0.25, 0.5, 0.6, 0.75, 1.0]
    transitions, ks = generate_kfe_transitions(zs_vec, zr_vec, k_steps)

    # Dekodierte Bilder + Konfidenz pro k
    dec_imgs, conf_matrix = [], []
    with torch.no_grad():
        for zk in transitions:
            dec_imgs.append(tanh_to_img(decoder(zk)).clamp(0, 1).cpu())
            conf_matrix.append(torch.softmax(classify_from_gap(model, zk), dim=1)[:, 1].cpu().numpy())
        cfe_probs = torch.softmax(classify_from_gap(model, zr_vec), dim=1)[:, 1].cpu().numpy()

    orig_probs_np = orig_probs[:n].detach().cpu().numpy()

    n_cols = 2 + len(k_steps)
    fig = plt.figure(figsize=(n_cols * 2.4, n * 3.2))
    gs  = gridspec.GridSpec(n, n_cols, figure=fig, hspace=0.55, wspace=0.25)

    k_labels = ['k=0\n(Quelle)', 'k=0.25', 'k=0.5\n(Grenze)',
                'k=0.6', 'k=0.75', 'k=1.0\n(Reflexion)']

    for i in range(n):
        img_np   = denormalise(images[i, 0].cpu()).numpy()
        src_lbl  = source_labels[i].item()
        cfe_lbl  = cfe_labels[i].item()
        true_lbl = int(labels[i])
        flipped  = int(cfe_probs[i] >= 0.5) == cfe_lbl

        ax0 = fig.add_subplot(gs[i, 0])
        ax0.imshow(img_np, cmap='gray')
        pred_col = 'limegreen' if src_lbl == true_lbl else 'tomato'
        p_src = orig_probs_np[i] if src_lbl == 1 else 1 - orig_probs_np[i]
        ax0.set_title(f'Original\nWahr: {class_names[true_lbl]}\n'
                      f'Pred: {class_names[src_lbl]} ({p_src:.1%})',
                      fontsize=7, color=pred_col)
        ax0.axis('off')

        for j, (k_lbl) in enumerate(k_labels):
            ax  = fig.add_subplot(gs[i, j + 1])
            ax.imshow(dec_imgs[j][i, 0].numpy(), cmap='gray', vmin=0, vmax=1)
            p_k       = conf_matrix[j][i]
            pred_at_k = class_names[int(p_k >= 0.5)]
            txt_col   = 'cyan' if (int(p_k >= 0.5) == cfe_lbl) else 'white'
            ax.text(0.5, 0.02, f'P(Kl.1)={p_k:.2f}\n{pred_at_k}',
                    transform=ax.transAxes, ha='center', va='bottom',
                    fontsize=6, color=txt_col,
                    bbox=dict(facecolor='black', alpha=0.5, pad=1))
            if abs(k_steps[j] - 0.5) < 0.01:
                for sp in ax.spines.values():
                    sp.set_edgecolor('cyan'); sp.set_linewidth(2)
            if k_steps[j] == 1.0:
                fc = 'limegreen' if flipped else 'tomato'
                for sp in ax.spines.values():
                    sp.set_edgecolor(fc); sp.set_linewidth(2)
            ax.set_title(k_lbl, fontsize=6.5)
            ax.set_xticks([]); ax.set_yticks([])

        ax_conf = fig.add_subplot(gs[i, -1])
        confs_i = [conf_matrix[j][i] for j in range(len(ks))]
        ax_conf.plot(ks, confs_i, 'o-', color='steelblue', lw=2, ms=4, label='P(Kl.1)')
        ax_conf.axhline(0.5, color='k', ls='--', lw=1, label='Grenze')
        ax_conf.axvline(0.5, color='cyan', ls=':', lw=1, label='k=0.5')
        ax_conf.set_xlim(-0.05, 1.05); ax_conf.set_ylim(-0.05, 1.1)
        ax_conf.set_xlabel('k', fontsize=7); ax_conf.set_ylabel('Konfidenz', fontsize=7)
        ax_conf.tick_params(labelsize=6); ax_conf.legend(fontsize=5.5, loc='upper left')
        fc = 'limegreen' if flipped else 'tomato'
        ax_conf.set_title(f'Konfidenz-Journey\n{class_names[src_lbl]} → {class_names[cfe_lbl]}\n'
                          f'{"✓ Gekippt" if flipped else "✗ Nicht gekippt"}',
                          fontsize=7, color=fc)

    plt.suptitle('Mirror-CFE — KFE-Übergang (k=0 bis k=1), dekodierte Bilder\n'
                 'Cyan = Grenze (k=0.5) | Grün/Rot = Reflexion (k=1)',
                 fontsize=11, y=1.01)
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Gespeichert → {save_path}')

print('Visualisierungsfunktion definiert ✓')


In [ ]:
sample_images, sample_labels, sample_fnames = next(iter(test_loader))

mirror_fv, cfe_labels, source_labels, orig_probs = compute_mirror_cfe(
    classifier, sample_images, DEVICE, num_iterations=LBFGS_ITERS
)

visualise_mirror_cfe(
    model         = classifier,
    decoder       = decoder,
    images        = sample_images,
    labels        = sample_labels,
    mirror_fv     = mirror_fv,
    cfe_labels    = cfe_labels,
    source_labels = source_labels,
    orig_probs    = orig_probs,
    class_names   = CLASS_NAMES,
    n_samples     = 4,
    save_path     = os.path.join(OUT_DIR, 'mirror_cfe_mnist_explanation.png')
)


## 14. Sanity Check — Flip Rate (Validity im Feature-Raum)
Misst, wie oft der reflektierte/verfeinerte Feature-Vektor laut Kopf tatsächlich zur
Zielklasse kippt — die zentrale Erfolgsmetrik der Mirror-CFE.

In [ ]:
mirror_fv_chk, cfe_labels_chk, source_labels_chk, _ = compute_mirror_cfe(
    classifier, sample_images, DEVICE, num_iterations=LBFGS_ITERS
)
cfe_probs_chk = predict_from_features(classifier, mirror_fv_chk).cpu()
cfe_preds_chk = (cfe_probs_chk >= 0.5).long()
flip_rate     = (cfe_preds_chk == cfe_labels_chk.cpu()).float().mean()

print(f'Batch-Größe   : {len(sample_labels)}')
print(f'Flip Rate     : {flip_rate:.2%}  (Ziel: >80%)')
print(f'Quellvorhers. : {source_labels_chk.tolist()}')
print(f'CFE-Vorhers.  : {cfe_preds_chk.tolist()}')
print(f'Zielklassen   : {cfe_labels_chk.tolist()}')
if flip_rate < 0.8:
    print('\n⚠ Flip Rate niedrig — LBFGS_ITERS erhöhen (z.B. 50)')


## 15. Metriken
Angepasste Version der Röntgen-/FCVE-Metriken (Proximity, Sparsity Rate, LPIPS, FID,
Validity, Denoised Validity, Coverage, Efficiency). EBPG entfällt (keine Bounding-Boxen
für MNIST). LPIPS/FID erwarten 3-Kanal-RGB — der Graukanal wird dafür auf 3 Kanäle
dupliziert. Alle Metriken laufen über die dekodierten CFE-Bilder (k=1, verfeinert).

In [ ]:
pip install lpips


In [ ]:
import time
from scipy import linalg as scipy_linalg

try:
    import lpips
    lpips_fn = lpips.LPIPS(net='squeeze').to(DEVICE)
    lpips_fn.eval()
    LPIPS_AVAILABLE = True
    print('LPIPS geladen ✓')
except ImportError:
    LPIPS_AVAILABLE = False
    print('⚠ lpips nicht verfügbar — pip install lpips')


def compute_mirror_cfe_full(classifier, decoder, images, device, num_iterations=20):
    """Mirror-CFE inkl. Dekodierung: gibt das CFE-Bild (k=1), Labels und Probs zurück."""
    mirror_fv, cfe_labels, source_labels, orig_probs = compute_mirror_cfe(
        classifier, images, device, num_iterations)
    z_cf = mirror_fv.flatten(1).detach()
    with torch.no_grad():
        cfe_image = decoder(z_cf)
        cfe_probs = torch.softmax(classify_from_gap(classifier, z_cf), dim=1)[:, 1]
    return cfe_image, cfe_labels, source_labels, orig_probs, cfe_probs


# ── 1. L1-Distanz (Proximity) ─────────────────────────────────────────────────
def compute_l1(orig_np, cfe_np):
    diffs = np.abs(orig_np - cfe_np)
    l1 = diffs.sum(axis=(1, 2, 3)) / (orig_np.shape[1] * orig_np.shape[2] * orig_np.shape[3])
    return float(l1.mean())

# ── 2. Sparsity Rate ──────────────────────────────────────────────────────────
def compute_sparsity_rate(orig_np, cfe_np, threshold=1e-4):
    diff = np.abs(orig_np - cfe_np).mean(axis=-1)
    changed = (diff > threshold).astype(float)
    return float(changed.mean(axis=(1, 2)).mean())

# ── 3. LPIPS ──────────────────────────────────────────────────────────────────
def _to_3ch(np_img_batch):
    if np_img_batch.shape[-1] == 1:
        return np.repeat(np_img_batch, 3, axis=-1)
    return np_img_batch

def compute_lpips(orig_np, cfe_np):
    if not LPIPS_AVAILABLE:
        return None
    orig_t = torch.tensor(_to_3ch(orig_np), dtype=torch.float32).permute(0, 3, 1, 2).to(DEVICE) * 2 - 1
    cfe_t  = torch.tensor(_to_3ch(cfe_np),  dtype=torch.float32).permute(0, 3, 1, 2).to(DEVICE) * 2 - 1
    with torch.no_grad():
        scores = lpips_fn(orig_t, cfe_t)
    return float(scores.mean().cpu())

# ── 4. FID — InceptionV3 ──────────────────────────────────────────────────────
from torchvision.models import inception_v3

@torch.no_grad()
def extract_inception_features(imgs_np, batch_size=32):
    imgs_np = _to_3ch(imgs_np)
    if not hasattr(extract_inception_features, '_model'):
        m = inception_v3(weights='DEFAULT', transform_input=False)
        m.fc = nn.Identity()
        extract_inception_features._model = m.eval().to(DEVICE)
    inc = extract_inception_features._model
    feats = []
    for i in range(0, len(imgs_np), batch_size):
        t = torch.tensor(imgs_np[i:i + batch_size], dtype=torch.float32).permute(0, 3, 1, 2).to(DEVICE)
        t = F.interpolate(t, size=(299, 299), mode='bilinear', align_corners=False)
        feats.append(inc(t).cpu().numpy())
    return np.concatenate(feats, axis=0)

def compute_fid(real_np, fake_np):
    rf, ff = extract_inception_features(real_np), extract_inception_features(fake_np)
    mu_r, mu_f = rf.mean(0), ff.mean(0)
    sig_r, sig_f = np.cov(rf, rowvar=False), np.cov(ff, rowvar=False)
    diff = mu_r - mu_f
    covmean, _ = scipy_linalg.sqrtm(sig_r @ sig_f, disp=False)
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    return float(diff @ diff + np.trace(sig_r + sig_f - 2 * covmean))

# ── 5. Validity & Denoised Validity ──────────────────────────────────────────
def compute_validity(model, cfe_imgs_tensor, cfe_labels, denoise_sigma=None):
    from torchvision.transforms.functional import gaussian_blur
    model.eval()
    cfe_imgs_tensor = cfe_imgs_tensor.to(DEVICE)
    if denoise_sigma is not None:
        ksz = max(int(denoise_sigma * 6) | 1, 3)
        cfe_imgs_tensor = gaussian_blur(cfe_imgs_tensor, kernel_size=[ksz, ksz],
                                        sigma=[denoise_sigma, denoise_sigma])
    cfe_norm = (cfe_imgs_tensor - 0.5) / 0.5
    with torch.no_grad():
        preds = model(cfe_norm).argmax(dim=1).cpu()
    return float((preds == cfe_labels.cpu()).float().mean())

# ── 6. Coverage ───────────────────────────────────────────────────────────────
def compute_coverage(classifier, decoder, images, n_runs=5, num_iterations=20):
    """Mirror-CFE ist deterministisch (L-BFGS aus geometrischer Initialisierung),
    daher std ~ 0 — das ist erwartet, kein Fehler."""
    images = images.to(DEVICE)
    valid = []
    for run in range(n_runs):
        torch.manual_seed(run)
        _, c_lbls, _, _, cfe_p = compute_mirror_cfe_full(
            classifier, decoder, images, DEVICE, num_iterations)
        preds = (cfe_p >= 0.5).long().cpu()
        valid.append((preds == c_lbls.cpu()).float().mean().item())
    return float(np.mean(valid)), float(np.std(valid))

print('Metrik-Funktionen definiert ✓')


## 16. Fester Eval-Satz
Ein fester, reproduzierbarer Subset des Test-Sets.

In [ ]:
EVAL_SEED = 42
N_EVAL_IMAGES = min(200, len(test_ds))

_rng = np.random.RandomState(EVAL_SEED)
_all_test_idx = np.arange(len(test_ds))
_rng.shuffle(_all_test_idx)
eval_indices = _all_test_idx[:N_EVAL_IMAGES].tolist()

eval_dataset = torch.utils.data.Subset(test_ds, eval_indices)
eval_loader = DataLoader(eval_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=True)
print(f'Eval-Satz: {len(eval_dataset)} Bilder (Seed {EVAL_SEED})')


## 17. Metriken berechnen

In [ ]:
DENOISE_SIGMA = 1.0

all_l1, all_sparsity, all_lpips = [], [], []
all_real_np, all_cfe_np = [], []
all_eval_imgs = []
all_cfe_tensor, all_cfe_labels = [], []
total_time = 0.0
n_images = 0

print(f'Berechne Metriken über {len(eval_dataset)} Bilder (fester Eval-Satz)...')
print('-' * 60)

for batch_imgs, batch_lbls, batch_fnames in tqdm(eval_loader, desc='Metrik-Evaluation'):

    t0 = time.time()
    b_cfe_img, b_cfe_lbl, b_src, _, b_cfe_p = compute_mirror_cfe_full(
        classifier, decoder, batch_imgs, DEVICE, num_iterations=LBFGS_ITERS
    )
    total_time += time.time() - t0
    n_images += len(batch_lbls)
    all_eval_imgs.append(batch_imgs.cpu())

    orig_np = denormalise(batch_imgs.cpu()).permute(0, 2, 3, 1).numpy()
    cfe_np  = tanh_to_img(b_cfe_img.detach().cpu()).permute(0, 2, 3, 1).numpy().clip(0, 1)

    all_l1.append(compute_l1(orig_np, cfe_np))
    all_sparsity.append(compute_sparsity_rate(orig_np, cfe_np))
    if LPIPS_AVAILABLE:
        all_lpips.append(compute_lpips(orig_np, cfe_np))

    all_real_np.append(orig_np)
    all_cfe_np.append(cfe_np)
    all_cfe_tensor.append(torch.tensor(cfe_np, dtype=torch.float32).permute(0, 3, 1, 2))
    all_cfe_labels.append(b_cfe_lbl.cpu())

# ── FID ───────────────────────────────────────────────────────────────────────
fid_score = compute_fid(np.concatenate(all_real_np, 0), np.concatenate(all_cfe_np, 0))

# ── Validity & Denoised Validity ─────────────────────────────────────────────
all_cfe_tensor_cat = torch.cat(all_cfe_tensor, dim=0)
all_cfe_labels_cat = torch.cat(all_cfe_labels, dim=0)
validity          = compute_validity(classifier, all_cfe_tensor_cat, all_cfe_labels_cat)
denoised_validity = compute_validity(classifier, all_cfe_tensor_cat, all_cfe_labels_cat,
                                     denoise_sigma=DENOISE_SIGMA)

# ── Coverage ──────────────────────────────────────────────────────────────────
print('Berechne Coverage (5 Runs über den festen Eval-Satz)...')
eval_imgs_cat = torch.cat(all_eval_imgs, dim=0)
coverage_mean, coverage_std = compute_coverage(
    classifier, decoder, eval_imgs_cat, n_runs=5, num_iterations=LBFGS_ITERS)

efficiency = total_time / n_images

print('\n' + '=' * 60)
print('METRIK-ERGEBNISSE — Mirror-CFE MNIST (7 ↔ 9), feinabgestimmter Decoder')
print('=' * 60)
print(f'\n── Proximity ───────────────────────────────────────────')
print(f'  L1-Distanz:           {np.mean(all_l1):.4f}  (↓ besser)')
print(f'\n── Interpretierbarkeit ─────────────────────────────────')
print(f'  Sparsity Rate:        {np.mean(all_sparsity):.4f}  (↓ besser)')
if LPIPS_AVAILABLE:
    print(f'  LPIPS (SqueezeNet):   {np.mean(all_lpips):.4f}  (↓ besser)')
else:
    print(f'  LPIPS:                nicht verfügbar')
print(f'\n── Plausibilität ───────────────────────────────────────')
print(f'  FID:                  {fid_score:.2f}   (↓ besser)')
print(f'\n── Funktionalität ──────────────────────────────────────')
print(f'  Validity:             {validity:.2%}  (↑ besser)')
print(f'  Denoised Validity:    {denoised_validity:.2%}  (↑ besser, σ={DENOISE_SIGMA})')
print(f'  Δ Validity:           {validity - denoised_validity:.2%}  (↓ besser = weniger adversarial)')
print(f'  Coverage:             {coverage_mean:.2%} ± {coverage_std:.2%}  (↑ besser)')
print(f'  Efficiency:           {efficiency:.4f}s / CF  (↓ besser)')
print('=' * 60)
